In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

In [2]:
df = pd.read_csv('naoEstacionaria\\Metro_Interstate_Traffic_Volume.csv')

COLUNA_DATA = 'date_time'
COLUNA_VALOR = 'traffic_volume'

df.dropna(subset=[COLUNA_VALOR], inplace=True)

df[COLUNA_DATA] = pd.to_datetime(df[COLUNA_DATA])
df = df.sort_values(COLUNA_DATA)
df.set_index(COLUNA_DATA, inplace=True)

df_daily = df[[COLUNA_VALOR]].resample('D').mean()

nulos = df_daily.isnull().sum()
if nulos.sum() > 0:
    df_daily = df_daily.interpolate(method='linear')
    print("x Dias vazios gerados no agrupamento foram preenchidos via interpolação para manter o índice contínuo.")

display(df_daily.head())

print("\n--- 4.1 Apresentação da Base ---")
print("Origem: UCI Machine Learning Repository (Metro Interstate Traffic Volume)")
print("Contexto: Volume de tráfego de veículos no sentido oeste da rodovia I-94 (Minneapolis-St Paul)")
print(f"Variável Temporal: {COLUNA_DATA}")
print(f"Variável Analisada: {COLUNA_VALOR}")
print("Frequência: Diária (Agrupado por Média)")
print(f"Período: de {df_daily.index[0].date()} a {df_daily.index[-1].date()}")
print(f"Quantidade de observações: {len(df_daily)}")
print("Possíveis problemas: Base original horária era densa demais e possuía quebras no timestamp; agrupada para diária resolvendo isso.")



x Dias vazios gerados no agrupamento foram preenchidos via interpolação para manter o índice contínuo.


,traffic_volume
date_time,
2012-10-02,4219.266667
2012-10-03,3317.250000
2012-10-04,3747.458333
2012-10-05,4242.545455
2012-10-06,3256.956522



--- 4.1 Apresentação da Base ---
Origem: UCI Machine Learning Repository (Metro Interstate Traffic Volume)
Contexto: Volume de tráfego de veículos no sentido oeste da rodovia I-94 (Minneapolis-St Paul)
Variável Temporal: date_time
Variável Analisada: traffic_volume
Frequência: Diária (Agrupado por Média)
Período: de 2012-10-02 a 2018-09-30
Quantidade de observações: 2190
Possíveis problemas: Base original horária era densa demais e possuía quebras no timestamp; agrupada para diária resolvendo isso.


In [3]:
# Divisão temporal (80/20)
train_size = int(len(df_daily) * 0.8)
treino = df_daily[COLUNA_VALOR].iloc[:train_size]
teste = df_daily[COLUNA_VALOR].iloc[train_size:]

print(f"Total de observações: {len(df_daily)}")
print(f"Treino: {len(treino)} semanas (de {treino.index[0].date()} a {treino.index[-1].date()})")
print(f"Teste: {len(teste)} semanas (de {teste.index[0].date()} a {teste.index[-1].date()})")


Total de observações: 2190
Treino: 1752 semanas (de 2012-10-02 a 2017-07-19)
Teste: 438 semanas (de 2017-07-20 a 2018-09-30)


1. Função – Média Histórica

Calcula a média de uma janela de valores anteriores.

In [4]:
# 1. MÉDIA HISTÓRICA 
def media_historica(train, test):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds

2. Função – Média Móvel Simples (SMA)

Calcula a média de uma janela de valores anteriores.

A **média móvel (moving average)** em um cálculo que projeta a **melhor janela** para calcular a média dos **valores consecutivos** e ir “andando” com essa janela ao longo.


In [5]:
def sma(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history[-janela:])

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_sma(train, test):
    resultados = []

    for janela in range(2, len(train) + 1):
        mae, preds = sma(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_sma(train, test):
    df_janelas = testar_janelas_sma(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

3. Função – Média Acumulada

Calcula a média de todos os valores até aquele momento.

In [6]:
def media_acumulada(train, test):
    history = list(train)
    preds = []

    for real in test:
        pred = np.mean(history)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds

4. Função – Média Móvel Exponencial (EMA)

Dá mais peso aos valores mais recentes.

In [7]:
def ema(train, test, alpha):
    history = list(train)

    ema_val = history[0]

    for valor in history[1:]:
        ema_val = alpha * valor + (1 - alpha) * ema_val

    preds = []

    for real in test:
        pred = ema_val

        preds.append(pred)

        ema_val = alpha * real + (1 - alpha) * ema_val
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_alphas_ema(train, test):
    resultados = []

    alphas = np.arange(0.1, 1.0, 0.1)

    for alpha in alphas:
        alpha = round(alpha, 1)

        mae, preds = ema(train, test, alpha)

        resultados.append({
            "Alpha": alpha,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_ema(train, test):
    df_alphas = testar_alphas_ema(train, test)

    melhor_alpha = df_alphas.iloc[0]["Alpha"]
    melhor_mae = df_alphas.iloc[0]["MAE"]
    melhores_preds = df_alphas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_alpha, df_alphas

5. Taxa de Variação (ROC)

Aqui o erro compara valor real vs taxa prevista anterior.

In [8]:
def taxa_variacao(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        x_t = history[-1]
        x_tk = history[-janela - 1]

        if x_tk == 0:
            pred = x_t
        else:
            taxa = (x_t - x_tk) / x_tk
            pred = x_t * (1 + taxa)

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_taxa_variacao(train, test):
    resultados = []

    for janela in range(1, len(train)):
        mae, preds = taxa_variacao(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_taxa_variacao(train, test):
    df_janelas = testar_janelas_taxa_variacao(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

6. Seasonal Naive

In [9]:
def seasonal_naive(train, test, periodo):
    history = list(train)
    preds = []

    for real in test:
        pred = history[-periodo]

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_periodos_seasonal_naive(train, test):
    resultados = []

    for periodo in range(2, len(train) + 1):
        mae, preds = seasonal_naive(train, test, periodo)

        resultados.append({
            "Periodo": periodo,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_seasonal_naive(train, test):
    df_periodos = testar_periodos_seasonal_naive(train, test)

    melhor_periodo = int(df_periodos.iloc[0]["Periodo"])
    melhor_mae = df_periodos.iloc[0]["MAE"]
    melhores_preds = df_periodos.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_periodo, df_periodos

7. DELTA

In [10]:
def delta(train, test, janela):
    history = list(train)
    preds = []

    for real in test:
        x_t = history[-1]
        x_tk = history[-janela - 1]

        delta = (x_t - x_tk) / janela
        pred = x_t + delta

        preds.append(pred)
        history.append(real)

    mae = mean_absolute_error(test, preds)

    return mae, preds


def testar_janelas_delta(train, test):
    resultados = []

    for janela in range(1, len(train)):
        mae, preds = delta(train, test, janela)

        resultados.append({
            "Janela": janela,
            "MAE": mae,
            "Previsoes": preds
        })

    df_resultados = pd.DataFrame(resultados)
    df_resultados = df_resultados.sort_values("MAE").reset_index(drop=True)

    return df_resultados


def melhor_delta(train, test):
    df_janelas = testar_janelas_delta(train, test)

    melhor_janela = int(df_janelas.iloc[0]["Janela"])
    melhor_mae = df_janelas.iloc[0]["MAE"]
    melhores_preds = df_janelas.iloc[0]["Previsoes"]

    return melhor_mae, melhores_preds, melhor_janela, df_janelas

In [11]:
# =========================================================
# RODANDO TODOS OS BASE MODELS
# =========================================================

mae_media_historica, pred_media_historica = media_historica(treino, teste)

mae_media_acumulada, pred_media_acumulada = media_acumulada(treino, teste)

mae_sma, pred_sma, melhor_janela_sma, df_sma_janelas = melhor_sma(treino, teste)

mae_ema, pred_ema, melhor_alpha_ema, df_ema_alphas = melhor_ema(treino, teste)

mae_taxa_variacao, pred_taxa_variacao, melhor_janela_taxa, df_taxa_janelas = melhor_taxa_variacao(treino, teste)

mae_seasonal_naive, pred_seasonal_naive, melhor_periodo_seasonal, df_seasonal_periodos = melhor_seasonal_naive(treino, teste)

mae_delta, pred_delta, melhor_janela_delta, df_delta_janelas = melhor_delta(treino, teste)

In [12]:
# =========================================================
# COMPARAÇÃO FINAL
# =========================================================

resultados_base_models = pd.DataFrame({
    "Modelo": [
        "Média Histórica",
        "Média Acumulada",
        "SMA",
        "EMA",
        "Taxa de Variação",
        "Seasonal Naive",
        "Delta"
    ],
    "Melhor Parâmetro": [
        "-",
        "-",
        f"janela={melhor_janela_sma}",
        f"alpha={melhor_alpha_ema}",
        f"janela={melhor_janela_taxa}",
        f"periodo={melhor_periodo_seasonal}",
        f"janela={melhor_janela_delta}"
    ],
    "MAE": [
        mae_media_historica,
        mae_media_acumulada,
        mae_sma,
        mae_ema,
        mae_taxa_variacao,
        mae_seasonal_naive,
        mae_delta
    ]
})

In [13]:
comparacao_previsoes = pd.DataFrame({
    "Real": teste.values,
    "Média Histórica": pred_media_historica,
    "Média Acumulada": pred_media_acumulada,
    "SMA": pred_sma,
    "EMA": pred_ema,
    "Taxa de Variação": pred_taxa_variacao,
    "Seasonal Naive": pred_seasonal_naive,
    "Delta": pred_delta
}, index=teste.index)

comparacao_previsoes

,Real,Média Histórica,Média Acumulada,SMA,EMA,Taxa de Variação,Seasonal Naive,Delta
date_time,,,,,,,,
2017-07-20,3760.666667,2985.914578,2985.914578,3276.699577,3342.592117,3187.619890,3851.125000,3344.780165
2017-07-21,3521.096774,2986.356535,2986.356535,3314.210795,3718.859212,3733.037819,3771.625000,3758.640000
2017-07-22,2810.560000,2986.661404,2986.661404,3306.888070,3540.873018,3251.078283,2938.916667,3494.921786
2017-07-23,2612.500000,2986.561062,2986.561062,3174.390362,2883.591302,2609.703907,2652.054054,2771.474425
2017-07-24,3356.703704,2986.348043,2986.348043,3148.963279,2639.609130,2814.932722,3375.447368,2593.213054
...,...,...,...,...,...,...,...,...
2018-09-26,3692.791667,3053.833900,3053.833900,3334.991954,3407.528485,3986.344751,3622.041667,3451.961835
2018-09-27,3777.360000,3054.126196,3054.126196,3371.696891,3664.265349,4396.732559,3794.560000,3691.072374
2018-09-28,3854.916667,3054.456893,3054.456893,3373.056623,3766.050535,3627.657750,3634.538462,3771.916749
